# Tensors

An in-depth introduction into `torch.Tensors` class

**Note**
- One dimensional tensor: Vector
- Two dimensional tensor: Matrix
- Above 2 dimensional tensor: Tensor

In [2]:
import torch
import math

## Creating Tensors

- **`torch.empty()`**

    Creates a tensor of a specified shape and data type without initializing the memory to any specific value. When called, PyTorch requests a block of contiguous memory from the system (or GPU) large enough to hold the requested dimensions and data type. The tensor values will simply be whatever bit patterns were already sitting in those allocated memory addresses.

In [3]:
x = torch.empty((2, 2, 4))
print(type(x))
print(x)

<class 'torch.Tensor'>
tensor([[[1.8899e+22, 1.8063e-42, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]]])


**When to use it**

- When you need a buffer memory that you plan to immediately overwrite.
- For very large tensors, writing zeros across gigabytes of memory takes non-trivial CPU/GPU cycles. `torch.empty()` completely eliminates that write cycle.

- **`torch.zeros()`**

    Unlike `torch.empty()`, it sets values (0) to each memory location.

In [4]:
print(torch.zeros((4, 3)))

tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


- **`torch.ones()`**

In [5]:
print(torch.ones((4, 3)))

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])


- **`torch.rand()`**

    Generates a tensor filled with random floating-point numbers drawn from a uniform distribution over the continuous interval [0, 1) (meaning 0 is inclusive and 1 is exclusive)

In [6]:
torch.manual_seed(1729) # random number generator (rng) seed
print(torch.rand((2, 3)))

tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])


In [7]:
torch.manual_seed(1729)
random1 = torch.rand(2, 3)
print(random1)

random2 = torch.rand(2, 3)
print(random2)

torch.manual_seed(1729)
random3 = torch.rand(2, 3)
print(random3)

random4 = torch.rand(2, 3)
print(random4)

tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])
tensor([[0.2332, 0.4047, 0.2162],
        [0.9927, 0.4128, 0.5938]])
tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])
tensor([[0.2332, 0.4047, 0.2162],
        [0.9927, 0.4128, 0.5938]])


**Note**

In the cell above, random1 and random 3 are similar tensors while random 2 and random4 are also similar. This happens because computers use a Pseudorandom Number Generator (PRNG), which is a fully deterministic mathematical algorithm rather than true randomness.


How the PRNG State Sequence Works

When you set a seed, you are fixing the starting point of an infinite, predefined sequence of numbers. Each random call consumes numbers from that sequence and advances the generator's internal state:

1. **`torch.manual_seed(1729)`**
Resets the generator back to the starting state.
2. **`random1 = torch.rand(2, 3)`**
Reads values 1 through 6 from the sequence.
3. **`random2 = torch.rand(2, 3)`**
Reads values 7 through 12 from the sequence.
4. **`torch.manual_seed(1729)`**
Resets the generator state.
5. **`random3 = torch.rand(2, 3)`**
Reads values 1 through 6 again.
6. **`random4 = torch.rand(2, 3)`**
Reads values 7 through 12 again.

Key Takeaway

A random seed does not just determine the single next value; it locks down the **entire ordered stream** of values. Any sequence of operations performed after setting that seed will produce the exact same sequence of numbers every single time.

In [8]:
# State manipulation
torch.manual_seed(1729)
_ = torch.rand(2, 3)  # Consumes elements 1 to 6

# Save the pointer position (at element 7)
saved_state = torch.get_rng_state()

rand_a = torch.rand(2, 3)  # Consumes elements 7 to 12

# Restore pointer back to element 7
torch.set_rng_state(saved_state)
rand_b = torch.rand(2, 3)  # Consumes elements 7 to 12 again

print(torch.equal(rand_a, rand_b))  # True

True


## Tensor Shapes

`.shape` is an attribute on a Tensor object which returns a `torch.Size` (a subclass of Python's standard tuple) object representing the dimensions of the tensor.

In [9]:
x = torch.empty(2, 2)
print(x.shape)
print(x)

empty_like_x = torch.empty_like(x) # initializes a tensor mathchig the shape, device and dtype of x
print(empty_like_x.shape)
print(empty_like_x)

zeros_like_x = torch.zeros_like(x)
print(zeros_like_x.shape)
print(zeros_like_x)

ones_like_x = torch.ones_like(x)
print(ones_like_x.shape)
print(ones_like_x)

rand_like_x = torch.rand_like(x)
print(rand_like_x.shape)
print(rand_like_x)

torch.Size([2, 2])
tensor([[2.3694e-38, 3.6013e-43],
        [0.0000e+00, 0.0000e+00]])
torch.Size([2, 2])
tensor([[0.0000e+00, 1.8980e+01],
        [2.3694e-38, 0.0000e+00]])
torch.Size([2, 2])
tensor([[0., 0.],
        [0., 0.]])
torch.Size([2, 2])
tensor([[1., 1.],
        [1., 1.]])
torch.Size([2, 2])
tensor([[0.6128, 0.1519],
        [0.0453, 0.5035]])


This property contains a list of the extent of each dimension of a tensor - in our case, `x` is a three-dimensional tensor with shape 2 x 2 x 3.

- **`torch.tensor()`**

    It is the most straightforward way to create a tensor if you already have data in a Python tuple or list.

In [10]:
some_constants = torch.tensor([[3.1415926, 2.71828], [1.61803, 0.0072897]])
print(some_constants)

some_integers = torch.tensor((2, 3, 5, 7, 11, 13, 17, 19))
print(some_integers)

more_integers = torch.tensor(((2, 4, 6), [3, 6, 9]))
print(more_integers)

tensor([[3.1416, 2.7183],
        [1.6180, 0.0073]])
tensor([ 2,  3,  5,  7, 11, 13, 17, 19])
tensor([[2, 4, 6],
        [3, 6, 9]])


Square brackets `[...]` (lists) and parenthesis `(...)` (tuples)can be used interchangeably and mixed together when passing data to `torch.tensor()`

In [11]:
# 1. Lists only (standard square brackets)
t1 = torch.tensor([[1, 2], [3, 4]])

# 2. Tuples only (parentheses)
t2 = torch.tensor(((1, 2), (3, 4)))

# 3. Outer tuple, inner lists
t3 = torch.tensor(([1, 2], [3, 4]))

# 4. Outer list, inner tuples
t4 = torch.tensor([(1, 2), (3, 4)])

# 5. Mixed at the same nested level
t5 = torch.tensor([(1, 2), [3, 4]])

Note: `torch.tensor()` creates a copy of the data i.e., changing the data has no effect on the tensor.

## Tensor Data Types

In [12]:
print(torch.get_default_dtype())

torch.float32


PyTorch's default floating-point type is `torch.float32` for factory operations (like `torch.empty()`, `torch.zeros()`, and `torch.rand()`).

However, it is invalid for general tensor creation like `torch.tensor([1, 2, 3])`, where PyTorch infers integer types (`torch.int64`) or boolean types (`torch.bool`) directly from the data.

Functions like `torch.from_numpy()` or `torch.as_tensor()` preserve the source array's data type (e.g., a float64 NumPy array becomes a `torch.float64` tensor).

In [13]:
a = torch.ones((2, 3), dtype=torch.int16)
print(a)

b = torch.rand((2, 3), dtype=torch.float64) * 20
print(b)

c = b.to(torch.int32)
print(c)

tensor([[1, 1, 1],
        [1, 1, 1]], dtype=torch.int16)
tensor([[19.6519, 10.8626,  2.1505],
        [19.6913,  0.9956,  1.4148]], dtype=torch.float64)
tensor([[19, 10,  2],
        [19,  0,  1]], dtype=torch.int32)


PyTorch provides a range of data types tailored for boolean logic, integer indexing, quantization, and mixed-precision deep learning.

| Data Type | Common Alias | Bytes / Bits | Range / Representation | Typical Use Case |
| --- | --- | --- | --- | --- |
| **`torch.bool`** | — | 1 byte (8 bits) | `True` or `False` | Masking, boolean indexing, logic comparisons |
| **`torch.int8`** | `torch.char` | 1 byte (8 bits) | $-128$ to $127$ | Model quantization (INT8 inference) |
| **`torch.uint8`** | `torch.byte` | 1 byte (8 bits) | $0$ to $255$ | Image pixel buffers, raw byte streams |
| **`torch.int16`** | `torch.short` | 2 bytes (16 bits) | $-32,768$ to $32,767$ | Memory-saving integer storage |
| **`torch.int32`** | `torch.int` | 4 bytes (32 bits) | $\approx -2.14 \times 10^9$ to $2.14 \times 10^9$ | C++/CUDA interop, low-level indexing |
| **`torch.int64`** | `torch.long` | 8 bytes (64 bits) | $\approx -9.22 \times 10^{18}$ to $9.22 \times 10^{18}$ | Tensor indices, target labels, token IDs |
| **`torch.half`** | `torch.float16` | 2 bytes (16 bits) | $\approx \pm 6.55 \times 10^4$ (10-bit precision) | GPU mixed-precision training/inference |
| **`torch.bfloat16`** | `torch.bfloat` | 2 bytes (16 bits) | $\approx \pm 3.39 \times 10^{38}$ (7-bit precision) | LLM training/inference (avoids underflow) |
| **`torch.float`** | `torch.float32` | 4 bytes (32 bits) | $\approx \pm 3.40 \times 10^{38}$ (23-bit precision) | Standard default for weights & activations |
| **`torch.double`** | `torch.float64` | 8 bytes (64 bits) | $\approx \pm 1.80 \times 10^{308}$ (52-bit precision) | Scientific computing, numerical stability checks |

**Boolean & Integer Types**

* **`torch.bool`**: Stores boolean flags. It takes 1 full byte per element in memory (for hardware addressing efficiency). Essential for operations like `tensor[tensor > 0]`.
* **`torch.int8`** & **`torch.uint8`**: 8-bit integers. `int8` is the standard for post-training quantization (PTQ) to shrink model sizes by 4× compared to `float32`. `uint8` is commonly used when reading raw image data (RGB values $0$–$255$).
* **`torch.int16`** & **`torch.int32`**: Rarely used directly in standard PyTorch pipelines, but `int32` is common in compiled CUDA code and custom C++ extensions where 64-bit pointers are not required.
* **`torch.int64` (`torch.long`)**: PyTorch's **default integer type**. It is required by functions like `nn.CrossEntropyLoss(pred, target)` (where target labels must be `torch.long`) and `nn.Embedding(indices)`.

**Floating-Point Types**

* **`torch.float` (`torch.float32`)**: Single-precision float. The standard workhorse of deep learning. It balances numerical stability with computational performance.
* **`torch.double` (`torch.float64`)**: Double-precision float. Extremely precise, but doubles memory usage and runs significantly slower on modern GPUs (which prioritize 16-bit and 32-bit math).
* **`torch.half` (`torch.float16`)** vs. **`torch.bfloat16`**:
* **`float16` (IEEE Half Precision):** Allocates 5 bits for exponent and 10 bits for precision. It has a very narrow dynamic range ($[-65504, 65504]$), making it prone to underflow/overflow. It requires **loss scaling** via `torch.cuda.amp.GradScaler`.
* **`bfloat16` (Brain Floating Point):** Allocates 8 bits for exponent (identical to `float32`) and 7 bits for precision. It matches the dynamic range of `float32` at half the memory footprint, eliminating the need for complex gradient scaling in modern LLM training.

## Math & Logic with Pytorch Tensors

In [14]:
ones = torch.zeros((2, 2)) + 1
twos = torch.ones((2, 2)) * 2
threes = (torch.ones((2, 2)) * 7 - 1) / 2
fours = twos ** 2
sqrt2s = twos ** (1/2)

print(ones)
print(twos)
print(threes)
print(fours)
print(sqrt2s)

tensor([[1., 1.],
        [1., 1.]])
tensor([[2., 2.],
        [2., 2.]])
tensor([[3., 3.],
        [3., 3.]])
tensor([[4., 4.],
        [4., 4.]])
tensor([[1.4142, 1.4142],
        [1.4142, 1.4142]])


When you apply a scalar (a single Python number or a 0-D tensor) to an $N$-dimensional tensor, PyTorch virtually expands the scalar to match the exact shape of the tensor without actually copying data or allocating extra memory. This is known as *Broadcasting*.

In [15]:
powers2 = twos ** torch.tensor([[1, 2], [3, 4]])
print(powers2)

fives = ones + fours
print(fives)

dozens = threes * fours
print(dozens)

tensor([[ 2.,  4.],
        [ 8., 16.]])
tensor([[5., 5.],
        [5., 5.]])
tensor([[12., 12.],
        [12., 12.]])


It's important to note here that all of the tensors in the previous code cell were of identical shape. The only scenario where mathematical operations are permitted for tensors of different shapes are those that follow the broadcasting rules.

**Broadcasting** in PyTorch is a mechanism that allows element-wise operations (such as addition, subtraction, or multiplication) on tensors with different shapes **without making unnecessary copies of data in memory**.

PyTorch automatically expands smaller dimensions to match larger dimensions virtually, providing significant speed and memory efficiency gains.

**The 3 Fundamental Rules of Broadcasting**

Two tensors are **broadcastable** if and only if they satisfy the following rules when aligned from right to left:

1. **Rule 1: Right-to-Left Alignment (Trailing Dimensions First)**
* PyTorch compares tensor dimensions starting from the **rightmost (trailing) dimension** and works leftward.
* If one tensor has fewer dimensions than the other, the missing leading dimensions are padded with $1$s.


2. **Rule 2: Dimension Compatibility (Equal or 1)**
* For every pair of aligned dimensions $i$, the sizes are compatible if:

$$\text{size}(A_i) == \text{size}(B_i) \quad \text{OR} \quad \text{size}(A_i) == 1 \quad \text{OR} \quad \text{size}(B_i) == 1$$


* If any pair of dimensions fails this condition (e.g., $3$ and $2$), PyTorch raises a `RuntimeError`.


3. **Rule 3: Output Shape Determination**
* The output tensor's shape at each dimension is the maximum of the input shapes along that dimension:

$$\text{size}(\text{Output}_i) = \max(\text{size}(A_i), \text{size}(B_i))$$


* Dimensions with a size of $1$ are virtually stretched ("broadcast") to match the larger size without consuming extra memory.

In [16]:
rand = torch.rand((2, 4))
doubled = rand * torch.ones((1, 4)) * 2

print(rand)
print(doubled)

tensor([[0.4115, 0.6839, 0.0703, 0.5105],
        [0.9451, 0.2359, 0.1979, 0.3327]])
tensor([[0.8230, 1.3677, 0.1405, 1.0210],
        [1.8901, 0.4717, 0.3959, 0.6655]])


In [17]:
A = torch.ones(3, 2)
B = torch.tensor([4, 5]) # Shape: (2,)

print(A-B)

tensor([[-3., -4.],
        [-3., -4.],
        [-3., -4.]])


In [18]:
a =     torch.ones(4, 3, 2)

b = a * torch.rand(   3, 2) # 3rd & 2nd dims identical to a, dim 1 absent
print(b)

c = a * torch.rand(   3, 1) # 3rd dim = 1, 2nd dim identical to a
print(c)

d = a * torch.rand(   1, 2) # 3rd dim identical to a, 2nd dim = 1

tensor([[[0.6146, 0.5999],
         [0.5013, 0.9397],
         [0.8656, 0.5207]],

        [[0.6146, 0.5999],
         [0.5013, 0.9397],
         [0.8656, 0.5207]],

        [[0.6146, 0.5999],
         [0.5013, 0.9397],
         [0.8656, 0.5207]],

        [[0.6146, 0.5999],
         [0.5013, 0.9397],
         [0.8656, 0.5207]]])
tensor([[[0.6865, 0.6865],
         [0.3614, 0.3614],
         [0.6493, 0.6493]],

        [[0.6865, 0.6865],
         [0.3614, 0.3614],
         [0.6493, 0.6493]],

        [[0.6865, 0.6865],
         [0.3614, 0.3614],
         [0.6493, 0.6493]],

        [[0.6865, 0.6865],
         [0.3614, 0.3614],
         [0.6493, 0.6493]]])


## Pytorch's Math Operations on Tensors

PyTorch tensors have over three hundred operations that can be performed on them.

In [21]:
# Common functions
a = torch.rand((2, 4)) * 2 - 1
print("Common functions:")
print(f'Base Tensor:\n{a}')
print(torch.abs(a))                 # computes the absolute value of each element
print(torch.ceil(a))                # rounds each floating-point value up
print(torch.floor(a))               # rounds each floating-point value down
print(torch.clamp(a, -0.5, 0.5))    # restricts all values to fall withing a specified interval [min, max]

# Trigonometric functions and their inverses
angles  = torch.tensor((0, math.pi/4, math.pi/2, 3*math.pi/4))
sines   = torch.sin(angles)
inverses = torch.asin(angles)
print('\nSine and arcsine:')
print(angles)
print(sines)
print(inverses)

# Bitwise operations
print('\nBitwise XOR:')
b = torch.tensor([1, 5, 11])
c = torch.tensor([2, 7, 10])
print(torch.bitwise_xor(b, c))

# Comparisons:
print('\nBroadcasted, element-wise equality comparison:')
d = torch.tensor([[1., 2.], [3., 4.]])
e = torch.ones(1, 2)  # many comparison ops support broadcasting!
print(torch.eq(d, e)) # returns a tensor of type bool

# Reductions:
print('\nReduction ops:')
print(torch.max(d))        # returns a single-element tensor
print(torch.max(d).item()) # extracts the value from the returned tensor
print(torch.mean(d))       # average
print(torch.std(d))        # standard deviation
print(torch.prod(d))       # product of all numbers
print(torch.unique(torch.tensor([1, 2, 1, 2, 1, 2]))) # filter unique elements

# Vector and linear algebra operations
v1 = torch.tensor([1., 0., 0.])         # x unit vector
v2 = torch.tensor([0., 1., 0.])         # y unit vector
m1 = torch.rand(2, 2)                   # random matrix
m2 = torch.tensor([[3., 0.], [0., 3.]]) # three times identity matrix

print('\nVectors & Matrices:')
print(torch.cross(v2, v1)) # negative of z unit vector (v1 x v2 == -v2 x v1)
print(m1)
m3 = torch.matmul(m1, m2)
print(m3)                  # 3 times m1
print(torch.svd(m3))       # singular value decomposition

Common functions:
Base Tensor:
tensor([[ 0.4750,  0.6657,  0.6888, -0.4118],
        [-0.2424, -0.0866, -0.8702,  0.3355]])
tensor([[0.4750, 0.6657, 0.6888, 0.4118],
        [0.2424, 0.0866, 0.8702, 0.3355]])
tensor([[1., 1., 1., -0.],
        [-0., -0., -0., 1.]])
tensor([[ 0.,  0.,  0., -1.],
        [-1., -1., -1.,  0.]])
tensor([[ 0.4750,  0.5000,  0.5000, -0.4118],
        [-0.2424, -0.0866, -0.5000,  0.3355]])

Sine and arcsine:
tensor([0.0000, 0.7854, 1.5708, 2.3562])
tensor([0.0000, 0.7071, 1.0000, 0.7071])
tensor([0.0000, 0.9033,    nan,    nan])

Bitwise XOR:
tensor([3, 2, 1])

Broadcasted, element-wise equality comparison:
tensor([[ True, False],
        [False, False]])

Reduction ops:
tensor(4.)
4.0
tensor(2.5000)
tensor(1.2910)
tensor(24.)
tensor([1, 2])

Vectors & Matrices:


C:\Users\ACER\AppData\Local\Temp\ipykernel_1596\2074140700.py:47: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\Cross.cpp:66.)
  print(torch.cross(v2, v1)) # negative of z unit vector (v1 x v2 == -v2 x v1)


tensor([ 0.,  0., -1.])
tensor([[0.7826, 0.1332],
        [0.0023, 0.4945]])
tensor([[2.3478, 0.3995],
        [0.0068, 1.4835]])
torch.return_types.svd(
U=tensor([[-0.9858, -0.1678],
        [-0.1678,  0.9858]]),
S=tensor([2.4032, 1.4482]),
V=tensor([[-0.9636, -0.2675],
        [-0.2675,  0.9636]]))


## Altering Tensors in Place

In [22]:
a = torch.tensor([0, math.pi / 4, math.pi / 2, 3 * math.pi / 4])
print('a:')
print(a)
print(torch.sin(a))   # this operation creates a new tensor in memory
print(a)              # a has not changed

b = torch.tensor([0, math.pi / 4, math.pi / 2, 3 * math.pi / 4])
print('\nb:')
print(b)
print(torch.sin_(b))  # note the underscore
print(b)              # b has changed

a:
tensor([0.0000, 0.7854, 1.5708, 2.3562])
tensor([0.0000, 0.7071, 1.0000, 0.7071])
tensor([0.0000, 0.7854, 1.5708, 2.3562])

b:
tensor([0.0000, 0.7854, 1.5708, 2.3562])
tensor([0.0000, 0.7071, 1.0000, 0.7071])
tensor([0.0000, 0.7071, 1.0000, 0.7071])


In [23]:
a = torch.ones(2, 2)
b = torch.rand(2, 2)

print('Before:')
print(a)
print(b)
print('\nAfter adding:')
print(a.add_(b))
print(a)
print(b)
print('\nAfter multiplying')
print(b.mul_(b))
print(b)

Before:
tensor([[1., 1.],
        [1., 1.]])
tensor([[0.3857, 0.9883],
        [0.4762, 0.7242]])

After adding:
tensor([[1.3857, 1.9883],
        [1.4762, 1.7242]])
tensor([[1.3857, 1.9883],
        [1.4762, 1.7242]])
tensor([[0.3857, 0.9883],
        [0.4762, 0.7242]])

After multiplying
tensor([[0.1488, 0.9768],
        [0.2268, 0.5245]])
tensor([[0.1488, 0.9768],
        [0.2268, 0.5245]])


Note that these in-place arithmetic functions are methods on the `torch.Tensor` object, not attached to the `torch` module like many other functions (e.g., `torch.sin()`). As you can see from `a.add_(b)`, *the calling tensor is the one that gets changed in place.*

There is another option for placing the result of a computation in an existing, allocated tensor. Many of the methods and functions we've seen so far - including creation methods! - have an `out` argument that lets you specify a tensor to receive the output. If the `out` tensor is the correct shape and `dtype`, this can happen without a new memory allocation:

In [32]:
a = torch.rand(2, 2)
b = torch.rand(2, 2)
c = torch.zeros(2, 2)
old_id = id(c)

print(c)
d = torch.matmul(a, b, out=c)
print(c)                # contents of c have changed

assert c is d           # test c & d are same object, not just containing equal values
assert id(c), old_id    # make sure that our new c is the same object as the old one

torch.rand(2, 2, out=c) # works for creation too!
print(c)                # c has changed again
assert id(c), old_id    # still the same object!

tensor([[0., 0.],
        [0., 0.]])
tensor([[0.5991, 0.7677],
        [0.8968, 1.2237]])
tensor([[0.4704, 0.6077],
        [0.4757, 0.5874]])


## Copying Tensors

In [27]:
a = torch.ones(2, 2)
b = a

a[0][1] = 561  # we change a...
print(b)       # ...and b is also altered

tensor([[  1., 561.],
        [  1.,   1.]])


But what if you want a separate copy of the data to work on? The `clone()` method is there for you:

In [30]:
a = torch.ones(2, 2)
b = a.clone()

assert b is not a      # different objects in memory...
print(torch.eq(a, b))  # ...but still with the same contents!

a[0][1] = 561          # a changes...
print(b)               # ...but b is still all ones

tensor([[True, True],
        [True, True]])
tensor([[1., 1.],
        [1., 1.]])


**`.clone()` when autograd is enabled**



## Moving to GPU

In [33]:
if torch.cuda.is_available():
    print('We have a GPU!')
else:
    print('Sorry, CPU only.')

Sorry, CPU only.


Once we've determined that one or more GPUs is available, we need to put our data someplace where the GPU can see it. Your CPU does computation on data in your computer's RAM. Your GPU has dedicated memory attached to it. Whenever you want to perform a computation on a device, you must move *all* the data needed for that computation to memory accessible by that device.

There are multiple ways to get your data onto your target device. You may do it at creation time:

In [34]:
if torch.cuda.is_available():
    gpu_rand = torch.rand(2, 2, device='cuda')
    print(gpu_rand)
else:
    print('Sorry, CPU only.')

Sorry, CPU only.


By default, new tensors are created on the CPU, so we have to specify when we want to create our tensor on the GPU with the optional `device` argument. You can see when we print the new tensor, PyTorch informs us which device it's on (if it's not on CPU).

You can query the number of GPUs with `torch.cuda.device_count()`. If you have more than one GPU, you can specify them by index: `device='cuda:0'`, `device='cuda:1'`, etc.

As a coding practice, specifying our devices everywhere with string constants is pretty fragile. In an ideal world, your code would perform robustly whether you're on CPU or GPU hardware. You can do this by creating a device handle that can be passed to your tensors instead of a string:

In [35]:
if torch.cuda.is_available():
    my_device = torch.device('cuda')
else:
    my_device = torch.device('cpu')
print('Device: {}'.format(my_device))

x = torch.rand(2, 2, device=my_device)
print(x)

Device: cpu
tensor([[0.4363, 0.6339],
        [0.3208, 0.4323]])


If you have an existing tensor living on one device, you can move it to another with the `to()` method. The following line of code creates a tensor on CPU, and moves it to whichever device handle you acquired in the previous cell.

In [36]:
y = torch.rand(2, 2)
y = y.to(my_device)

## Manipulating Tensor Shapes

**Changing the number dimensions**

One case where you might need to change the number of dimensions is passing a single instance of input to your model. PyTorch models generally expect *batches* of input.

For example, imagine having a model that works on 3 x 226 x 226 images - a 226-pixel square with 3 color channels. When you load and transform it, you'll get a tensor of shape `(3, 226, 226)`. Your model, though, is expecting input of shape `(N, 3, 226, 226)`, where `N` is the number of images in the batch. So how do you make a batch of one?

The `unsqueeze()` method adds a dimension of extent 1. `unsqueeze(0)` adds it as a new zeroth dimension - now you have a batch of one!

In [42]:
a = torch.rand((3, 266, 266))
b = a.unsqueeze(0)

print(a.shape)
print(b.shape)

torch.Size([3, 266, 266])
torch.Size([1, 3, 266, 266])


So if that's *un*squeezing? What do we mean by squeezing? We're taking advantage of the fact that any dimension of extent 1 *does not* change the number of elements in the tensor.

In [43]:
c = torch.rand(1, 1, 1, 1, 1)
print(c)

tensor([[[[[0.5904]]]]])


Continuing the example above, let's say the model's output is a 20-element vector for each input. You would then expect the output to have shape `(N, 20)`, where `N` is the number of instances in the input batch. That means that for our single-input batch, we'll get an output of shape `(1, 20)`.

What if you want to do some *non-batched* computation with that output - something that's just expecting a 20-element vector?

In [44]:
a = torch.rand(1, 20)
print(a.shape)
print(a)

b = a.squeeze(0)
print(b.shape)
print(b)

c = torch.rand(2, 2)
print(c.shape)

d = c.squeeze(0)
print(d.shape)

torch.Size([1, 20])
tensor([[0.5228, 0.0362, 0.1565, 0.9298, 0.9474, 0.7500, 0.3399, 0.3254, 0.7353,
         0.5337, 0.5409, 0.2617, 0.5171, 0.1547, 0.4295, 0.7799, 0.7801, 0.2308,
         0.9744, 0.0910]])
torch.Size([20])
tensor([0.5228, 0.0362, 0.1565, 0.9298, 0.9474, 0.7500, 0.3399, 0.3254, 0.7353,
        0.5337, 0.5409, 0.2617, 0.5171, 0.1547, 0.4295, 0.7799, 0.7801, 0.2308,
        0.9744, 0.0910])
torch.Size([2, 2])
torch.Size([2, 2])


Another place you might use `unsqueeze()` is to ease broadcasting.

In [45]:
a = torch.ones(4, 3, 2)
b = torch.rand(   3)     # trying to multiply a * b will give a runtime error
c = b.unsqueeze(1)       # change to a 2-dimensional tensor, adding new dim at the end
print(c.shape)
print(a * c)             # broadcasting works again!

torch.Size([3, 1])
tensor([[[0.2387, 0.2387],
         [0.5278, 0.5278],
         [0.1863, 0.1863]],

        [[0.2387, 0.2387],
         [0.5278, 0.5278],
         [0.1863, 0.1863]],

        [[0.2387, 0.2387],
         [0.5278, 0.5278],
         [0.1863, 0.1863]],

        [[0.2387, 0.2387],
         [0.5278, 0.5278],
         [0.1863, 0.1863]]])


The `squeeze()` and `unsqueeze()` methods also have in-place versions, `squeeze_()` and `unsqueeze_()`:

In [46]:
batch_me = torch.rand(3, 226, 226)
print(batch_me.shape)
batch_me.unsqueeze_(0)
print(batch_me.shape)

torch.Size([3, 226, 226])
torch.Size([1, 3, 226, 226])


Sometimes you'll want to change the shape of a tensor more radically, while still preserving the number of elements and their contents. One case where this happens is at the interface between a convolutional layer of a model and a linear layer of the model - this is common in image classification models. A convolution kernel will yield an output tensor of shape *features x width x height,* but the following linear layer expects a 1-dimensional input. `reshape()` will do this for you, provided that the dimensions you request yield the same number of elements as the input tensor has:

In [47]:
output3d = torch.rand(6, 20, 20)
print(output3d.shape)

input1d = output3d.reshape(6 * 20 * 20)
print(input1d.shape)

# can also call it as a method on the torch module:
print(torch.reshape(output3d, (6 * 20 * 20,)).shape)

torch.Size([6, 20, 20])
torch.Size([2400])
torch.Size([2400])


*(Note: The `(6 * 20 * 20,)` argument in the final line of the cell above is because PyTorch expects a **tuple** when specifying a tensor shape - but when the shape is the first argument of a method, it lets us cheat and just use a series of integers. Here, we had to add the parentheses and comma to convince the method that this is really a one-element tuple.)*

When it can, `reshape()` will return a *view* on the tensor to be changed - that is, a separate tensor object looking at the same underlying region of memory. *This is important:* That means any change made to the source tensor will be reflected in the view on that tensor, unless you `clone()` it.

There *are* conditions, beyond the scope of this introduction, where `reshape()` has to return a tensor carrying a copy of the data. For more information, see the [docs](https://pytorch.org/docs/stable/torch.html#torch.reshape).

## Numpy Bridge

If you have existing ML or scientific code with data stored in NumPy ndarrays, you may wish to express that same data as PyTorch tensors, whether to take advantage of PyTorch's GPU acceleration, or its efficient abstractions for building ML models. It's easy to switch between ndarrays and PyTorch tensors:

In [37]:
import numpy as np

numpy_array = np.ones((2, 3))
print(numpy_array)

pytorch_tensor = torch.from_numpy(numpy_array)
print(pytorch_tensor)

[[1. 1. 1.]
 [1. 1. 1.]]
tensor([[1., 1., 1.],
        [1., 1., 1.]], dtype=torch.float64)


PyTorch creates a tensor of the same shape and containing the same data as the NumPy array, going so far as to keep NumPy's default 64-bit float data type.

The conversion can just as easily go the other way:

In [38]:
pytorch_rand = torch.rand(2, 3)
print(pytorch_rand)

numpy_rand = pytorch_rand.numpy()
print(numpy_rand)

tensor([[0.3977, 0.3132, 0.6331],
        [0.8222, 0.2652, 0.7328]])
[[0.39766717 0.31324917 0.6330963 ]
 [0.8221878  0.26522762 0.732837  ]]


It is important to know that these converted objects are using *the same underlying memory* as their source objects, meaning that changes to one are reflected in the other:

In [39]:
numpy_array[1, 1] = 23
print(pytorch_tensor)

pytorch_rand[1, 1] = 17
print(numpy_rand)

tensor([[ 1.,  1.,  1.],
        [ 1., 23.,  1.]], dtype=torch.float64)
[[ 0.39766717  0.31324917  0.6330963 ]
 [ 0.8221878  17.          0.732837  ]]


In [40]:
print(torch.rand(1, 1, 1, 1, 1))

tensor([[[[[0.2126]]]]])
